<a href="https://colab.research.google.com/github/SotaYoshida/Lecture_DataScience/blob/master/Python_chapter3_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 相関・回帰分析

[この章の目的]
初歩的な相関分析と回帰分析がPythonで少し出来るようになる。

資料: 授業のスライド (C-Learning上で公開予定)

補足資料:講義ノート
https://drive.google.com/file/d/1ZKi8DJFSg00xir1IoEQiw3z9vxmejeCv/view


## 相関分析

1年生前期の必修科目[DS入門]では多くの学科で[相関分析]を学習しました。  
「エクセルでの相関分析をやってない」という方は、適宜講義ノートのXX章を参照してください。

解析データが２種類のデータだけなら、プログラムを使うありがたみはそれほど感じられないと思いますが「多くのデータ間の相関関係を系統的に調べたい」あるいは「複数年度に渡るデータを解析したい」となると、Excelはデータが大きくなるとすぐに挙動が重くなってしまうため精神によくありません(私見)ので、Pythonで扱うのがオススメです。

以下では、簡単な例の相関分析を扱って、時間に余裕があれば、やや発展的な内容としてxlsx(csv)形式のファイルに格納された大量のデータを扱ってみることにします。




(編集中)

## 回帰分析

回帰とはモデル関数(自分が立てたモデルを表現する関数)とデータとの齟齬を最小化するように
モデル関数の係数を決定することです。

データとの齟齬を表現する方法はいくつかありますが、以下では最もポピュラーな最小二乗法(誤差の二乗和を最初化する)方法を採用することとします。


まず回帰を学ぶために、適当なデータを生成しておきましょう

In [ ]:
import numpy as np
def create_toy_data(sample_size, std):
    x = np.linspace(0, 1, sample_size)
    t = np.sin(2*np.pi*x) + np.random.normal(scale=std, size=x.shape)                                                                                                                 
    return x, t

x,y = create_toy_data(10,1.e-1)

これをグラフにしてみると...

In [ ]:
import matplotlib.pyplot as plt
fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(111)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.scatter(x, y, facecolor="none", edgecolor="b", s=50, label="Data")
ax.legend()
plt.show()
plt.close()

こんな感じ。

このデータを、p次元の多項式(p=0,1,2,...)でフィッティングすることを考えてみましょう。

p次元のフィッティングはnumpyを利用すれば一瞬で行えます。  
(本来は自分の手で書くのが教育的ですが...全学部向けの授業なのでとりあえずライブラリを利用することにします。他にもscikit-learnなどのライブラリもより高度なfit関数が使えます。)  

たとえば今のデータを３次式でフィットしたければ、以下のようにします

In [ ]:
xp = np.linspace(0, 1, 500) ## 多項式をplotするためのxの値を準備(グラフをなめらかにするために、0から1までの間の500点を等間隔に取る)
p=3
yp = np.poly1d(np.polyfit(x, y, p))(xp)

fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(111)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.scatter(x, y, facecolor="none", edgecolor="b", s=50, label="Data")
ax.plot(xp, yp,label="p=3")
ax.legend()
plt.show()
plt.close()

```np.polyfit(x, y, p)```では、データのx,yの値と多項式の次元pを引数として与えることで、
p次の多項式でデータ(x,y)をfitしなさい(つまり、p次までの係数を"最適化"しなさい)という指令を与えています.

```np.poly1d( np.polyfit(x, y, p) )(xp)```では、fitしたp次元の係数をもつ多項式にxp(今は500点)の値を代入して、対応するyの値を返します。  
上のコードはこの返り値をypという変数に格納して、グラフの描画に使いました。


さて、一般にpを増やせば(多項式の次元が上がれば)、より複雑な関数を表現することができます。(p次の多項式はp-1次の多項式を特別な場合として含むため)
pを複数変えながら比較した図を作ってみましょう。方法は```p```に関するループを回すだけです。

In [ ]:
ps = [0,1,3,6,9]
ys = []
xp = np.linspace(0, 1, 500) ## 多項式をplotするためのxの値を準備
for p in ps:
    ys += [np.poly1d(np.polyfit(x, y, p))(xp)]

fig = plt.figure(figsize=(10,4))
ax = fig.add_subplot(111)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.scatter(x, y, facecolor="none", edgecolor="b", s=50, label="Data")
for i in range(len(ps)):
    ax.plot(xp, ys[i],label="p="+str(ps[i]))
ax.legend()
plt.show()
plt.close()

> 注: 今の場合、コードをよく見るとデータはsin関数に適当なノイズを足したものだということが分かります。  
解析の手法を学ぶ際には、このように答えを知っている状態からはじめて、手法がうまくデータを説明しているかどうかを検証したりします。一見ズルっぽいですが、重要なプロセスです。  
一方で、背後にある"真の関数"が分かっている状況は現実のデータ解析では非常に稀ですし、「興味のあるデータが、人間がよく知っている単純な式(有限次元の多項式や指数関数)で完全に表現できる道理はない」ということも抑えておくべき重要な点です.真の関数というのは一般に[神のみぞ知る]で、人間ができることは出来るだけ尤もらしい関数を見つけ、その背後にあるメカニズム(の主要部分)を解明することです.  
一般に、関数をどんどん複雑なものにしていくにつれて、表現できるデータの幅は大きく拡がります。その一方で、用意した関数がデータに過度に適合するあまり、未知の点での値の予測精度が著しく損なわれている危険性があります。  
このことを予言能力がない (汎化性能がない, データに過適合している) と言ったりします。  
データの背後にあるメカニズムが何かを考えたり、理論的な解析をして初めて、回帰に用いる関数の
妥当性が検証できるという点に注意しましょう。


## 新型コロナウイルス感染症の陽性者数の推移に関して
回帰分析に関連して、この話題も取り上げておきましょう。  

我々が現実世界で観測することのできる種々の[値]というのは、何らかの関数$f(x)$の、ある$x$での値と言えるかと思います。
テレビで見るコロナウイルスの感染者数の推移だって、日付に対する関数になっていますよね。

ただし一般に物事の背景にある"真の関数"というのは多次元の変数に対する関数になっているはずで、コロナウイルスの感染者数の推移だって、単なる時間に対する１変数の関数であるはずなどがありません。  

日付に対して陽性者数の推移をプロットして「このままだと指数関数的に増加する」という予想を立てることは簡単ですが、一般に物事は多様な要素が絡み合っています。
たとえば検査数や我々の外出自粛や国・都道府県ごとの取り組み・政策、ウイルスの変異,その他様々な要素に左右されるはずです。

我々人間がグラフにして理解できるのはたかだか３次元(３つの変数がある状況)までです。  
言い換えれば、人間は物事を理解するときに本来D次元(D>>3, Dは３よりずっと大きい)の変数で定義される関数を、3次元以下に射影した「影」をみて理解しようとする生き物だということは、意識しておくべきでしょう。


「私が考えた最強のモデル」を使って感染者数を予測して、危険を煽ったり、あるいは逆に「過度に心配する必要がない」などと主張したりする専門家・非専門家が増えていますが、未来を正しく予想することは、感染症の専門家でも、数理モデルの専門家でも、AI(?)でも、全ての専門家でも出来はしません。  

事態が収束したあとに「私のモデルはこんなに正しかった」という人も現れることでしょう。ですが、それは極めて高い蓋然性で「たまたま」です。無限の数の関数を考えれば、データに適合するものが存在してもおかしくはありませんよね。

重要なことは、「感染者数や死者の数は予想は出来ないが意識次第である程度コントロールできる余地が残されている」、つまりそれらの数理モデルは感染者の数を予測するためではなく「より良い政策を皆で見つけるため」に使うべきだ、ということではないでしょうか？

そして何より、モデルを立てて終わり、ではなく検証することが重要です。